# Nivel III: Espacio de Estados Diferenciable (ST-GNN + Filtro de Kalman)

- **Ecuación de estado (dinámica biológica latente):**
  $$x_t = x_{t-1} + f_\theta(x_{t-1}, u_{t-1}, \hat{A}) + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$
- **Ecuación de observación (reporte epidemiológico con ruido/rezagos):**
  $$z_t = \log(1 + y_t) = x_t + v_t, \quad v_t \sim \mathcal{N}(0, R)$$
- **Filtro de Kalman Diferenciable en PyTorch:**
  - Aprende conjuntamente la red espacio-temporal $f_\theta$, la covarianza del proceso $Q$ y la covarianza de medición $R$ por cantón.
  - Separa el **estado latente filtrado** ($x_{t|t}$) del ruido administrativo de reporte (no todos los cantones tienden a entregar el reporte a tiempo).
  - Entrega **pronósticos puntuales e intervalos de credibilidad al 95%** basados en la varianza predictiva $S_t = P_{t|t-1} + R$.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import mlflow

In [2]:
sns.set_theme(style="whitegrid", context="notebook")

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
os.chdir(ROOT_DIR)

df = pd.read_csv(ROOT_DIR / "data/processed/dengue_panel_clean.csv")
df = df.sort_values(["time_idx", "canton"]).reset_index(drop=True)
cantons = np.sort(df["canton"].unique())
N_NODES = len(cantons)
T_TOTAL = df["time_idx"].nunique()

## 1. Grafo Funcional $W$ y Tensores Secuenciales

In [3]:
# Grafo funcional Top-K=8 calculado únicamente en train <= 2023
train_only = df[df["year"] <= 2023].copy()
train_only["log_casos"] = np.log1p(train_only["casos"])
piv_tr = train_only.pivot(index="time_idx", columns="canton", values="log_casos")[cantons]
corr_tr = piv_tr.corr().fillna(0.0).to_numpy(dtype=np.float32)
np.fill_diagonal(corr_tr, 0.0)

TOP_K = 8
W = np.zeros_like(corr_tr)
for i in range(N_NODES):
    for j in np.argsort(corr_tr[i])[-TOP_K:]:
        if corr_tr[i, j] > 0:
            W[i, j] = corr_tr[i, j]
W = np.maximum(W, W.T)
A_tilde = W + np.eye(N_NODES, dtype=np.float32)
d_inv = np.power(A_tilde.sum(axis=1), -0.5)
A_norm_t = torch.tensor((d_inv[:, None] * A_tilde) * d_inv[None, :], dtype=torch.float32, device=device)

# Covariables exógenas U_all: (T, N, F_exog)
exog_cols = ["temp_prom", "tmin_mean", "precip_total", "ndvi", "nino34ssta", "dengue", "week_sin", "week_cos", "urb"]
scale_cols = ["temp_prom", "tmin_mean", "precip_total", "ndvi", "nino34ssta", "dengue", "urb"]
tr_mask_rows = df["year"] <= 2023
df_scaled = df.copy()
for c in scale_cols:
    mu = df.loc[tr_mask_rows, c].mean()
    sd = df.loc[tr_mask_rows, c].std() + 1e-6
    df_scaled[c] = (df[c] - mu) / sd

U_all = np.zeros((T_TOTAL, N_NODES, len(exog_cols)), dtype=np.float32)
for f_i, col in enumerate(exog_cols):
    U_all[:, :, f_i] = df_scaled.pivot(index="time_idx", columns="canton", values=col)[cantons].to_numpy(dtype=np.float32)

Y_raw = df.pivot(index="time_idx", columns="canton", values="casos")[cantons].to_numpy(dtype=np.float32)
Z_obs = np.log1p(Y_raw)  # Observación en escala log1p
years_by_t = df.groupby("time_idx")["year"].first().sort_index().to_numpy()

U_t = torch.tensor(U_all, dtype=torch.float32, device=device)
Z_t = torch.tensor(Z_obs, dtype=torch.float32, device=device)

tr_end = int(np.where(years_by_t <= 2023)[0][-1]) + 1
va_start = int(np.where(years_by_t == 2024)[0][0])
te_start = int(np.where(years_by_t == 2025)[0][0])
print(f"Índices temporales -> Train: 0..{tr_end-1} | Val: {va_start}..{te_start-1} | Test: {te_start}..{T_TOTAL-1}")

## 2. Modelo Espacio-Temporal + Filtro de Kalman Diferenciable (`KalmanSTGNN`)

En cada semana $t = 1, \dots, T-1$:
1. **Predicción (Prior):** usa el estado latente filtrado previo $x_{t-1|t-1}$, las covariables $u_{t-1}$ y la adyacencia híbrida $\hat{A}_{\text{mix}}$ para proyectar $\hat{x}_{t|t-1}$ y su varianza $P_{t|t-1} = P_{t-1|t-1} + Q$.
2. **Innovación y Verosimilitud:** compara $z_t$ contra $\hat{x}_{t|t-1}$ con varianza total $S_t = P_{t|t-1} + R$.
3. **Actualización (Posterior):** aplica la ganancia de Kalman $K_t = P_{t|t-1} / S_t$ para obtener el estado biológico limpio $x_{t|t}$ y $P_{t|t} = (1 - K_t) P_{t|t-1}$.

In [4]:
class KalmanSTGNN(nn.Module):
    def __init__(self, exog_dim, hidden_dim=48, num_nodes=82, emb_dim=12, dropout=0.15):
        super().__init__()
        self.node_emb = nn.Parameter(torch.randn(num_nodes, emb_dim) * 0.05)
        self.e1 = nn.Parameter(torch.randn(num_nodes, 8) * 0.1)
        self.e2 = nn.Parameter(torch.randn(num_nodes, 8) * 0.1)

        in_dim = 1 + exog_dim + emb_dim  # [estado_latente, exógenas, node_emb]
        self.self_lin = nn.Linear(in_dim, hidden_dim)
        self.graph_lin = nn.Linear(in_dim, hidden_dim)
        self.gru_cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.delta_head = nn.Linear(hidden_dim, 1)
        nn.init.zeros_(self.delta_head.weight)
        nn.init.zeros_(self.delta_head.bias)

        # Covarianzas diagonales aprendibles por cantón: proceso (Q) y medición/reporte (R)
        self.log_q = nn.Parameter(torch.full((num_nodes,), -2.0))
        self.log_r = nn.Parameter(torch.full((num_nodes,), -1.5))
        self.hidden_dim = hidden_dim

    def forward(self, z_seq, u_seq, a_norm, t_limit=None):
        # z_seq: (T, N), u_seq: (T, N, F_exog)
        T = z_seq.shape[0] if t_limit is None else t_limit
        N = z_seq.shape[1]

        a_adapt = F.softmax(F.relu(self.e1 @ self.e2.T), dim=-1)
        a_mix = 0.7 * a_norm + 0.3 * a_adapt

        Q = F.softplus(self.log_q).clamp(min=1e-4, max=2.0)  # (N,)
        R = F.softplus(self.log_r).clamp(min=1e-4, max=2.0)  # (N,)

        # Inicialización en t=0
        x_post = z_seq[0]  # (N,)
        p_post = Q.clone()
        h_gru = torch.zeros(N, self.hidden_dim, device=z_seq.device)

        pred_means, pred_vars, filtered_means = [], [], []

        for t in range(1, T):
            inp = torch.cat([x_post.unsqueeze(-1), u_seq[t - 1], self.node_emb], dim=-1)  # (N, in_dim)
            h_self = F.relu(self.self_lin(inp))
            h_nei = F.relu(self.graph_lin(a_mix @ inp))
            h_in = self.drop(h_self + h_nei)
            h_gru = self.gru_cell(h_in, h_gru)

            # 1. Paso de Predicción (Prior del Filtro de Kalman)
            delta = 0.5 * torch.tanh(self.delta_head(h_gru).squeeze(-1))
            x_prior = torch.clamp(x_post + delta, min=0.0, max=6.5)
            p_prior = p_post + Q
            s_var = p_prior + R  # Varianza predictiva total de la observación

            pred_means.append(x_prior)
            pred_vars.append(s_var)

            # 2. Paso de Actualización (Posterior tras observar z_seq[t])
            kalman_gain = p_prior / s_var
            x_post = x_prior + kalman_gain * (z_seq[t] - x_prior)
            p_post = (1.0 - kalman_gain) * p_prior
            filtered_means.append(x_post)

        return torch.stack(pred_means), torch.stack(pred_vars), torch.stack(filtered_means), Q, R


def gaussian_nll_loss(z_true, mean, var):
    return 0.5 * (torch.log(2.0 * np.pi * var) + ((z_true - mean) ** 2) / var).mean()

## 3. Entrenamiento End-to-End del Filtro de Kalman y Evaluación Probabilística

In [5]:
def compute_count_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    rmsle = float(np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)))
    wape = float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)))
    p = f"{prefix}_" if prefix else ""
    return {f"{p}mae": round(mae, 4), f"{p}rmse": round(rmse, 4), f"{p}rmsle": round(rmsle, 4), f"{p}wape": round(wape, 4)}

HIDDEN_DIM = 48
LR = 5e-3
EPOCHS = 400
PATIENCE = 35

k_model = KalmanSTGNN(exog_dim=len(exog_cols), hidden_dim=HIDDEN_DIM, num_nodes=N_NODES).to(device)
optimizer = torch.optim.AdamW(k_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_rmsle = float("inf")
best_state = None
wait = 0
history = {"train_nll": [], "val_nll": []}

for epoch in range(1, EPOCHS + 1):
    k_model.train()
    optimizer.zero_grad()
    # Entrenamos estrictamente en train (t=1..tr_end-1)
    p_mean_tr, p_var_tr, _, _, _ = k_model(Z_t, U_t, A_norm_t, t_limit=tr_end)
    loss_tr = gaussian_nll_loss(Z_t[1:tr_end], p_mean_tr, p_var_tr)
    loss_tr.backward()
    torch.nn.utils.clip_grad_norm_(k_model.parameters(), 2.0)
    optimizer.step()
    scheduler.step()

    k_model.eval()
    with torch.no_grad():
        p_mean_full, p_var_full, _, _, _ = k_model(Z_t, U_t, A_norm_t, t_limit=te_start)
        # p_mean_full[idx] corresponde a t = idx + 1
        va_slice = slice(va_start - 1, te_start - 1)
        loss_va = float(gaussian_nll_loss(Z_t[va_start:te_start], p_mean_full[va_slice], p_var_full[va_slice]).item())
        y_va_pred = np.clip(np.expm1(p_mean_full[va_slice].cpu().numpy()), 0.0, None)
        m_va = compute_count_metrics(Y_raw[va_start:te_start].ravel(), y_va_pred.ravel(), "val")

    history["train_nll"].append(float(loss_tr.item()))
    history["val_nll"].append(loss_va)

    if m_va["val_rmsle"] < best_val_rmsle:
        best_val_rmsle = m_va["val_rmsle"]
        best_state = {k: v.cpu().clone() for k, v in k_model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping en época {epoch} | mejor val_rmsle={best_val_rmsle:.4f}")
            break

    if epoch % 20 == 0 or epoch == 1:
        print(f"Época {epoch:03d} | train_nll={loss_tr.item():.4f} | val_nll={loss_va:.4f} | val_mae={m_va['val_mae']} | val_rmsle={m_va['val_rmsle']}")

k_model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
k_model.eval()
with torch.no_grad():
    p_mean_all, p_var_all, filt_all, Q_est, R_est = k_model(Z_t, U_t, A_norm_t)

pred_mean_np = p_mean_all.cpu().numpy()
pred_std_np = np.sqrt(p_var_all.cpu().numpy())
filt_np = np.clip(np.expm1(filt_all.cpu().numpy()), 0.0, None)

y_pred_all = np.clip(np.expm1(pred_mean_np), 0.0, None)
ci_low_all = np.clip(np.expm1(pred_mean_np - 1.96 * pred_std_np), 0.0, None)
ci_high_all = np.clip(np.expm1(pred_mean_np + 1.96 * pred_std_np), 0.0, None)

va_sl = slice(va_start - 1, te_start - 1)
te_sl = slice(te_start - 1, T_TOTAL - 1)

val_metrics = compute_count_metrics(Y_raw[va_start:te_start].ravel(), y_pred_all[va_sl].ravel(), "val")
test_metrics = compute_count_metrics(Y_raw[te_start:T_TOTAL].ravel(), y_pred_all[te_sl].ravel(), "test")

# Cobertura empírica del intervalo de credibilidad al 95% (PICP)
val_picp95 = float(np.mean((Y_raw[va_start:te_start] >= ci_low_all[va_sl]) & (Y_raw[va_start:te_start] <= ci_high_all[va_sl])))
test_picp95 = float(np.mean((Y_raw[te_start:T_TOTAL] >= ci_low_all[te_sl]) & (Y_raw[te_start:T_TOTAL] <= ci_high_all[te_sl])))

print("\nKalman-STGNN ->", val_metrics, test_metrics)
print(f"Cobertura IC 95% -> Val: {val_picp95*100:.2f}% | Test: {test_picp95*100:.2f}%")
print(f"Ruido promedio aprendido -> Proceso Q: {float(Q_est.mean()):.4f} | Medición/Reporte R: {float(R_est.mean()):.4f}")

In [14]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10))

axes[0].plot(history["train_nll"], label="Train Kalman-NLL")
axes[0].plot(history["val_nll"], label="Val Kalman-NLL")
axes[0].set_title("Curva de Entrenamiento Kalman-STGNN")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Gaussian NLL (log1p)")
axes[0].grid(False)
axes[0].legend()

t_eval = np.arange(va_start, T_TOTAL)
eval_sl = slice(va_start - 1, T_TOTAL - 1)

for ax, cid in zip(axes[1:], [101, 305]):
    c_idx = int(np.where(cantons == cid)[0][0])
    ax.plot(t_eval, Y_raw[va_start:T_TOTAL, c_idx], color="black", lw=1.8, label="Casos reales")
    ax.plot(t_eval, y_pred_all[eval_sl, c_idx], color="#1f77b4", lw=1.6, label="Pronóstico Kalman-STGNN")
    ax.plot(t_eval, filt_np[eval_sl, c_idx], color="#2ca02c", ls="--", lw=1.3, alpha=0.85, label="Estado latente filtrado")
    ax.fill_between(t_eval, ci_low_all[eval_sl, c_idx], ci_high_all[eval_sl, c_idx], color="#1f77b4", alpha=0.18, label="IC 95%")
    ax.axvline(te_start, color="gray", ls=":", label="Inicio Test (2025)")
    ax.set_title(f"Kalman-STGNN con Intervalos de Credibilidad 95% — Cantón {cid}")
    ax.set_ylabel("Casos")
    ax.grid(False)
    ax.set_ylim(0, 800)
    ax.legend(loc="upper right", ncol=3)

axes[-1].set_xlabel("time_idx")
fig.tight_layout()
fig_kal = ROOT_DIR / "data/processed/eda/11_nivel3_kalman_stgnn.png"
fig.savefig(fig_kal, dpi=150)
plt.show()

En el último modelo implementado, se ha buscado la menera de cómo hacer que el modelo pueda comprender de mejor manera el comportamiento de los datos, esto cambiando parámetros, sin embargo este no ha tenido un acercamiento comparado a Arimax e inclusive, a TS-GNN sin Kalman.

Mi recomendación para este modelaje, les recomiendo usar TS-GNN con Kalman, devido a que Kalman tiene una capaciad de comprender y extraer información de ambos aspectos, tanto del modelo teorico, así como de los datos empiricos, como los que Kalman corrigió que los datos se fueran exponencial por una breve suvida de la información dada.

Finalmente, recomiendo experimentar con modelos genéticos, para aproximar mejor los resultados.

In [7]:
ckpt_path = ROOT_DIR / "models/checkpoints/kalman_stgnn_best.pt"
ckpt_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(best_state, ckpt_path)

mlruns_dir = (ROOT_DIR / "mlruns").resolve()
mlflow.set_tracking_uri(f"sqlite:///{(mlruns_dir / 'mlflow.db').as_posix()}")
mlflow.set_experiment("dengue_cr_spatiotemporal")

with mlflow.start_run(run_name="03_kalman_stgnn_state_space"):
    mlflow.set_tags({"stage": "nivel3_kalman_stgnn", "graph_type": f"functional_corr_top{TOP_K}", "split_policy": "train<=2023|val=2024|test=2025"})
    mlflow.log_params({"hidden_dim": HIDDEN_DIM, "lr": LR, "top_k_graph": TOP_K, "loss": "kalman_gaussian_nll_log1p"})
    mlflow.log_metrics({
        **val_metrics, **test_metrics,
        "val_picp_95": round(val_picp95, 4),
        "test_picp_95": round(test_picp95, 4),
        "mean_Q": round(float(Q_est.mean()), 4),
        "mean_R": round(float(R_est.mean()), 4),
    })
    mlflow.log_artifact(str(fig_kal), artifact_path="nivel3_plots")
    mlflow.log_artifact(str(ckpt_path), artifact_path="checkpoints")

exp_runs = mlflow.search_runs(experiment_names=["dengue_cr_spatiotemporal"])
exp_runs = exp_runs[exp_runs["tags.stage"] != "nivel1_tuning"]
cols_show = [c for c in ["tags.mlflow.runName", "tags.stage", "metrics.val_mae", "metrics.val_rmsle", "metrics.val_wape", "metrics.test_mae", "metrics.test_rmsle", "metrics.test_wape"] if c in exp_runs.columns]
display(exp_runs[cols_show].sort_values("metrics.val_rmsle", na_position="last"))